In [1]:
from dbfread import DBF
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
output_dir = Path("../dataset/Kfintech AMCs NAVs 09052025/processed/fcodes_csv")
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
dbf_files = list(Path("../dataset/Kfintech AMCs NAVs 09052025/").glob("*.dbf"))

In [4]:
for file in dbf_files:
    print(f"Processing: {file.name}")

    dbf_data = DBF(file, encoding="cp1252")
    df_temp = pd.DataFrame(dbf_data)

    for fcode, group in df_temp.groupby("FCODE"):
        output_file = output_dir / f"{fcode}.csv"

        group.to_csv(
            output_file,
            mode="a",
            header=not output_file.exists(),
            index=False
        )

    del df_temp

print("DBF → CSV extraction complete.")

Processing: W10N8611.dbf
Processing: W11N2151.dbf
Processing: W12N2598.dbf
Processing: W13N339.dbf
Processing: W14N1470.dbf
Processing: W15N4207.dbf
Processing: W16N5742.dbf
Processing: W17N468.dbf
Processing: W18N5916.dbf
Processing: W19N1447.dbf
Processing: W1N2662.dbf
Processing: W20N6789.dbf
Processing: W21N30.dbf
Processing: W22N9069.dbf
Processing: W23N6002.dbf
Processing: W24N6032.dbf
Processing: W25N5867.dbf
Processing: W26N002.dbf
Processing: W27N7735.dbf
Processing: W28N3979.dbf
Processing: W29N5107.dbf
Processing: W2N5399.dbf
Processing: W30N693.dbf
Processing: W31N5788.dbf
Processing: W32N9315.dbf
Processing: W33N4553.dbf
Processing: W34N4780.dbf
Processing: W35N5911.dbf
Processing: W36N746.dbf
Processing: W37N1489.dbf
Processing: W38N1019.dbf
Processing: W39N460.dbf
Processing: W3N5130.dbf
Processing: W40N8909.dbf
Processing: W41N2140.dbf
Processing: W42N4378.dbf
Processing: W43N961.dbf
Processing: W44N4750.dbf
Processing: W45N9081.dbf
Processing: W46N9228.dbf
Processing: 

In [5]:
csv_files = list(output_dir.glob("*.csv"))

total_csv_records = 0

for file in csv_files:
    count = len(pd.read_csv(file, dtype=str))
    total_csv_records += count

print("FCODE CSV files:", len(csv_files))
print("Total CSV records:", total_csv_records)

FCODE CSV files: 15387
Total CSV records: 16093603


In [6]:
total_dbf_records = 0

for file in dbf_files:
    dbf_data = DBF(file, encoding="cp1252")
    count = len(dbf_data)

    print(file.name, ":", count)

    total_dbf_records += count

print("\nTotal DBF records:", total_dbf_records)

W10N8611.dbf : 300000
W11N2151.dbf : 300000
W12N2598.dbf : 300000
W13N339.dbf : 300000
W14N1470.dbf : 300000
W15N4207.dbf : 300000
W16N5742.dbf : 300000
W17N468.dbf : 300000
W18N5916.dbf : 300000
W19N1447.dbf : 300000
W1N2662.dbf : 300000
W20N6789.dbf : 300000
W21N30.dbf : 300000
W22N9069.dbf : 300000
W23N6002.dbf : 300000
W24N6032.dbf : 300000
W25N5867.dbf : 300000
W26N002.dbf : 300000
W27N7735.dbf : 300000
W28N3979.dbf : 300000
W29N5107.dbf : 300000
W2N5399.dbf : 300000
W30N693.dbf : 300000
W31N5788.dbf : 300000
W32N9315.dbf : 300000
W33N4553.dbf : 300000
W34N4780.dbf : 300000
W35N5911.dbf : 300000
W36N746.dbf : 300000
W37N1489.dbf : 300000
W38N1019.dbf : 300000
W39N460.dbf : 300000
W3N5130.dbf : 300000
W40N8909.dbf : 300000
W41N2140.dbf : 300000
W42N4378.dbf : 300000
W43N961.dbf : 300000
W44N4750.dbf : 300000
W45N9081.dbf : 300000
W46N9228.dbf : 300000
W47N668.dbf : 300000
W48N408.dbf : 300000
W49N7630.dbf : 300000
W4N353.dbf : 300000
W50N5671.dbf : 300000
W51N7502.dbf : 300000
W52N

In [7]:
print("Match:", total_dbf_records == total_csv_records)
print("Difference:", total_dbf_records - total_csv_records)

Match: True
Difference: 0


In [8]:
csv_dir = output_dir
parquet_dir = Path("../dataset/Kfintech AMCs NAVs 09052025/processed/fcode_desc")
metadata_file = Path("../dataset/Kfintech AMCs NAVs 09052025/processed/fcode_desc_metadata.parquet")

In [9]:
parquet_dir.mkdir(parents=True, exist_ok=True)

In [10]:
csv_files = sorted(csv_dir.glob("*.csv"))

print(f"FCODE CSV files: {len(csv_files)}")

FCODE CSV files: 15387


In [11]:
unique_id = 1
metadata = []

In [12]:
def load_csv(file):
    return pd.read_csv(file)

In [13]:
def set_data_types(df):
    df["NAVDATE"] = pd.to_datetime(df["NAVDATE"])
    df["NAV"] = pd.to_numeric(df["NAV"], errors="coerce")
    df["RPOP"] = pd.to_numeric(df["RPOP"], errors="coerce")
    df["PPOP"] = pd.to_numeric(df["PPOP"], errors="coerce")
    df["CRDATE"] = pd.to_datetime(df["CRDATE"])
    df["CRTIME"] = pd.to_numeric(df["CRTIME"], errors="coerce")

    return df

In [14]:
def group_by_fund_desc(df):
    return df.groupby("FUNDDESC")

In [15]:
def assign_id():
    global unique_id

    group_id = unique_id
    unique_id += 1

    return group_id

In [16]:
def create_parquet(group, group_id, parquet_dir, metadata):
    fcode = group["FCODE"].iloc[0]

    # Observation data
    output = group[
        ["NAVDATE", "NAV", "RPOP", "PPOP"]
    ].copy()

    output["FCODE_DESC_ID"] = group_id

    output_file = parquet_dir / f"{group_id:06d}-{fcode}.parquet"

    output.to_parquet(
        output_file,
        index=False
    )

    # Metadata
    row = group.iloc[0]

    metadata.append({
        "FCODE_DESC_ID": group_id,
        "FCODE": fcode,
        "FUND": row["FUND"],
        "SCHEME": row["SCHEME"],
        "SCHEMEISIN": row["SCHEMEISIN"],
        "CRDATE": row["CRDATE"],
        "CRTIME": row["CRTIME"],
        "FUNDDESC": row["FUNDDESC"]
    })

In [17]:
for file in csv_files:

    df = load_csv(file)

    df = set_data_types(df)

    groups = group_by_fund_desc(df)

    for funddesc, group in groups:

        group_id = assign_id()

        create_parquet(
            group,
            group_id,
            parquet_dir,
            metadata
        )

    del df

In [18]:
metadata_df = pd.DataFrame(metadata)

# Metadata string columns
string_columns = [
    "FCODE",
    "FUND",
    "SCHEME",
    "SCHEMEISIN",
    "FUNDDESC"
]

for col in string_columns:
    metadata_df[col] = metadata_df[col].astype(str)

# Metadata date columns
metadata_df["CRDATE"] = pd.to_datetime(
    metadata_df["CRDATE"]
)

# Metadata time column
metadata_df["CRTIME"] = pd.to_numeric(
    metadata_df["CRTIME"],
    errors="coerce"
)

# Make sure the output directory exists
metadata_file.parent.mkdir(parents=True, exist_ok=True)

metadata_df.to_parquet(
    metadata_file,
    index=False
)

print("Conversion complete.")
print(f"Parquet files: {len(list(parquet_dir.glob('*.parquet')))}")
print(f"Metadata records: {len(metadata_df)}")

Conversion complete.
Parquet files: 15594
Metadata records: 15594


In [19]:
# Count CSV records
csv_records = 0

for file in csv_files:
    csv_records += len(pd.read_csv(file, dtype=str))


# Count Parquet records
parquet_records = 0

for file in parquet_dir.glob("*.parquet"):
    parquet_records += len(pd.read_parquet(file))


print("CSV records:     ", csv_records)
print("Parquet records: ", parquet_records)
print("Difference:      ", csv_records - parquet_records)
print("Match:            ", csv_records == parquet_records)

CSV records:      16093603
Parquet records:  16093603
Difference:       0
Match:             True


In [20]:
# Clean up the output directory after checking

import shutil
shutil.rmtree(output_dir)

print("All CSV files deleted.")

All CSV files deleted.


In [21]:
parquet_files = sorted(parquet_dir.glob("*.parquet"))

all_nav_file = Path("../dataset/Kfintech AMCs NAVs 09052025/processed/all_funds.parquet")

writer = None

for file in parquet_files:
    table = pq.read_table(file)

    if writer is None:
        writer = pq.ParquetWriter(
            all_nav_file,
            table.schema
        )

    writer.write_table(table)

writer.close()

print("Combined Parquet created.")

Combined Parquet created.


In [22]:
import pandas as pd

# Load combined Parquet
all_nav = pd.read_parquet(all_nav_file)

# Count records
combined_records = len(all_nav)

print("Combined records:", combined_records)

Combined records: 16093603


In [23]:
print("Individual records:", parquet_records)
print("Combined records:  ", combined_records)
print("Difference:        ", parquet_records - combined_records)
print("Match:             ", parquet_records == combined_records)

Individual records: 16093603
Combined records:   16093603
Difference:         0
Match:              True
